# Flow Matching on 32x32 Mazes

This notebook trains the same timestep-conditioned U-Net as the DDPM experiment, but with conditional flow matching. A straight interpolation connects Gaussian noise at `t=0` to the embedded maze at `t=1`; the network predicts the path velocity. Sampling integrates the learned ODE from noise to data.

The maze is represented by two fixed orthonormal pixel embeddings (`wall=0`, `passage=1`). The final continuous image is greedily decoded by dot product. Run cells manually; training may take a while.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import copy
import importlib
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from plotly.subplots import make_subplots

from _shared.maze_data import generate_dataset, solvability_rate
import _shared.model as model_module

importlib.reload(model_module)
from _shared.model import (
    Denoiser,
    decode_probabilities,
    embed_binary,
    make_binary_embeddings,
)

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Synthetic Dataset

Randomized DFS generates perfect mazes. `0` means wall; `1` means passage.

In [2]:
N_MAZES = 12_000
VAL_FRACTION = 0.1
mazes = generate_dataset(N_MAZES, seed=SEED)
n_val = int(N_MAZES * VAL_FRACTION)
train_mazes = torch.from_numpy(mazes[:-n_val].copy()).long()
val_mazes = torch.from_numpy(mazes[-n_val:].copy()).long()

fig = go.Figure(
    go.Heatmap(
        z=mazes[0],
        zmin=0,
        zmax=1,
        colorscale=[[0, "black"], [1, "white"]],
        showscale=False,
    )
)
fig.update_layout(
    title="Synthetic maze example",
    width=420,
    height=420,
    yaxis=dict(autorange="reversed", scaleanchor="x"),
)
fig.show()

## 2. Forward Flow Path

For a clean embedded maze `x_data` and independent Gaussian noise `z`, define the conditional path

`x_t = (1 - t) * z + t * x_data`, with `t` sampled uniformly from `[0, 1]`.

Its target velocity is constant along the path: `v_target = x_data - z`. This panel decodes the continuous states using the same fixed-temperature embedding dot-product visualization at every time; the visualization does not alter the path.

In [3]:
EMBEDDING_DIM = 2
TEMPERATURE = 0.5
binary_embeddings = make_binary_embeddings(EMBEDDING_DIM, DEVICE)

preview_maze = torch.from_numpy(mazes[0].copy()).long().unsqueeze(0).to(DEVICE)
x_data = embed_binary(preview_maze, binary_embeddings)
preview_noise = torch.randn_like(x_data)
preview_times = [0.0, 0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0]
with torch.no_grad():
    preview_frames = []
    for t in preview_times:
        t_batch = torch.full((1,), t, device=DEVICE)
        x_t = (1.0 - t_batch[:, None, None, None]) * preview_noise + t_batch[
            :, None, None, None
        ] * x_data
        probabilities = decode_probabilities(x_t, binary_embeddings, TEMPERATURE)
        preview_frames.append(probabilities[0, 1].cpu().numpy())

path_fig = make_subplots(
    rows=1,
    cols=len(preview_times),
    subplot_titles=[f"t={t:g}" for t in preview_times],
)
for col, frame in enumerate(preview_frames, start=1):
    path_fig.add_trace(
        go.Heatmap(
            z=frame,
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=1,
        col=col,
    )
path_fig.update_layout(
    title="Linear flow path: black = p(0), white = p(1)",
    width=1700,
    height=360,
)
path_fig.update_xaxes(showticklabels=False)
path_fig.update_yaxes(showticklabels=False, autorange="reversed")
path_fig.show()

## 3. Model Architecture

The U-Net structure is shared with the DDPM notebook: input shape `(B, 2, 32, 32)`, base channels `64` with encoder widths `64 -> 128 -> 256`, residual blocks conditioned by a `128`-dimensional time embedding, and a two-channel output. Here the output represents velocity `v(x_t, t)`, not Gaussian noise. The normalized flow time is scaled to `[0, 100]` before sinusoidal embedding so the existing time encoder receives a useful range. See `notebooks/_shared/model.py` for the implementation.

In [4]:
BASE_CHANNELS = 64
TIME_DIM = 128
TIME_EMBEDDING_SCALE = 100.0

initial_model = Denoiser(
    embedding_dim=EMBEDDING_DIM,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
).to(DEVICE)

# Clone one initialization so the objective is the only planned difference.
models = {
    "velocity": initial_model,
    "endpoint_residual": copy.deepcopy(initial_model),
    "random_interval": copy.deepcopy(initial_model),
}
MODEL_LABELS = {
    "velocity": "0: Velocity MSE",
    "endpoint_residual": "1: Endpoint-residual MSE",
    "random_interval": "2: Random-interval MSE",
}

## 4. Compare Three Flow-Matching Objectives

All three models use the same U-Net, initial weights, maze batches, Gaussian noise, and sampled $t_0$. They always predict a velocity $v_\theta(x_{t_0},t_0)$.

1. **Velocity MSE:** $\operatorname{MSE}(x_1-z, v_\theta)$.
2. **Endpoint-residual MSE:** $\operatorname{MSE}(x_1-x_{t_0}, (1-t_0)v_\theta)$.
3. **Random-interval MSE:** sample $\Delta t$ per example and use $\operatorname{MSE}(x_{t_1}-x_{t_0}, \Delta t\,v_\theta)$, where $t_1=t_0+\Delta t$.

The second objective weights velocity errors by $(1-t_0)^2$; the third weights them by $\Delta t^2$. Their raw loss magnitudes therefore are not directly comparable. Solvability after sampling provides a common outcome measure. Validation uses fixed noise and times so its curves are less noisy.


In [5]:
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 2e-4
PLOT_EVERY = 1
DT_MIN = 0.01
DT_MAX = 0.2

train_loader = DataLoader(
    TensorDataset(train_mazes), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
val_loader = DataLoader(TensorDataset(val_mazes), batch_size=BATCH_SIZE, shuffle=False)
optimizers = {
    name: torch.optim.AdamW(candidate.parameters(), lr=LEARNING_RATE)
    for name, candidate in models.items()
}


def sample_training_context(maze_batch, generator=None):
    """Share a maze/noise/time context across methods 0 and 1; add random dt for 2."""
    x_data = embed_binary(maze_batch.to(DEVICE), binary_embeddings)
    noise = torch.randn(
        x_data.shape, generator=generator, device=DEVICE, dtype=x_data.dtype
    )
    batch_size = x_data.shape[0]
    t0 = (1.0 - DT_MIN) * torch.rand(
        batch_size, generator=generator, device=DEVICE, dtype=x_data.dtype
    )
    t0_image = t0[:, None, None, None]
    x_t0 = (1.0 - t0_image) * noise + t0_image * x_data

    remaining = (1.0 - t0).clamp_min(torch.finfo(x_data.dtype).eps)
    dt_upper = torch.minimum(torch.full_like(t0, DT_MAX), remaining)
    dt_lower = torch.minimum(torch.full_like(t0, DT_MIN), dt_upper)
    dt = dt_lower + torch.rand(
        batch_size, generator=generator, device=DEVICE, dtype=x_data.dtype
    ) * (dt_upper - dt_lower)
    t1 = (t0 + dt).clamp(max=1.0)
    t1_image = t1[:, None, None, None]
    x_t1 = (1.0 - t1_image) * noise + t1_image * x_data
    return x_data, noise, t0, x_t0, dt, x_t1


def objective_losses(context, candidate_models):
    x_data, noise, t0, x_t0, dt, x_t1 = context
    velocity_target = x_data - noise
    residual_target = x_data - x_t0
    interval_target = x_t1 - x_t0
    t0_scale = (1.0 - t0)[:, None, None, None]
    dt_scale = dt[:, None, None, None]
    losses = {}
    for name, candidate in candidate_models.items():
        predicted_velocity = candidate(x_t0, t0 * TIME_EMBEDDING_SCALE)
        if name == "velocity":
            losses[name] = F.mse_loss(predicted_velocity, velocity_target)
        elif name == "endpoint_residual":
            losses[name] = F.mse_loss(t0_scale * predicted_velocity, residual_target)
        else:
            losses[name] = F.mse_loss(dt_scale * predicted_velocity, interval_target)
    return losses


@torch.no_grad()
def validation_losses():
    # Recreate the same validation contexts every epoch for a stable comparison.
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + 401)
    was_training = {name: candidate.training for name, candidate in models.items()}
    for candidate in models.values():
        candidate.eval()
    totals = {name: [] for name in models}
    try:
        for (maze_batch,) in val_loader:
            context = sample_training_context(maze_batch, generator)
            for name, value in objective_losses(context, models).items():
                totals[name].append(value.item())
        return {name: float(np.mean(values)) for name, values in totals.items()}
    finally:
        for name, candidate in models.items():
            candidate.train(was_training[name])


train_history = {name: [] for name in models}
val_history = {name: [] for name in models}
loss_fig = go.Figure()
method_colors = {
    "velocity": "#4263eb",
    "endpoint_residual": "#e76f00",
    "random_interval": "#2a9d8f",
}
for name in models:
    loss_fig.add_scatter(
        name=f"{MODEL_LABELS[name]} - train",
        mode="lines",
        line=dict(color=method_colors[name], width=2),
    )
    loss_fig.add_scatter(
        name=f"{MODEL_LABELS[name]} - validation",
        mode="lines",
        line=dict(color=method_colors[name], width=2, dash="dot"),
    )
loss_fig.update_layout(
    title="Three flow-matching objectives",
    xaxis_title="Epoch",
    yaxis_title="MSE (log scale; objective scales differ)",
    yaxis_type="log",
    width=1100,
    height=560,
    legend=dict(orientation="h", y=-0.2),
)
for epoch in range(1, EPOCHS + 1):
    for candidate in models.values():
        candidate.train()
    epoch_values = {name: [] for name in models}
    for (maze_batch,) in train_loader:
        context = sample_training_context(maze_batch)
        for name, candidate in models.items():
            loss = objective_losses(context, {name: candidate})[name]
            optimizers[name].zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(candidate.parameters(), max_norm=1.0)
            optimizers[name].step()
            epoch_values[name].append(loss.item())

    for name in models:
        train_history[name].append(float(np.mean(epoch_values[name])))
    validation = validation_losses()
    for name in models:
        val_history[name].append(validation[name])

    if epoch % PLOT_EVERY == 0 or epoch == 1:
        for method_index, name in enumerate(models):
            train_trace = 2 * method_index
            val_trace = train_trace + 1
            loss_fig.data[train_trace].x = list(range(1, epoch + 1))
            loss_fig.data[train_trace].y = np.maximum(train_history[name], 1e-12)
            loss_fig.data[val_trace].x = list(range(1, epoch + 1))
            loss_fig.data[val_trace].y = np.maximum(val_history[name], 1e-12)
    loss_fig.update_layout(title=f"Three objectives | epoch {epoch}/{EPOCHS}")
    clear_output(wait=True)
    display(loss_fig)

## 5. Save the Three Models


In [6]:
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
checkpoint_path = ARTIFACTS / "maze_flow_matching_objectives.pt"
torch.save(
    {
        "models_state_dict": {
            name: candidate.state_dict() for name, candidate in models.items()
        },
        "binary_embeddings": binary_embeddings.detach().cpu(),
        "config": {
            "embedding_dim": EMBEDDING_DIM,
            "base_channels": BASE_CHANNELS,
            "time_dim": TIME_DIM,
            "time_embedding_scale": TIME_EMBEDDING_SCALE,
            "path": "linear_gaussian_to_data",
            "objectives": [
                "mse_velocity_x1_minus_noise",
                "mse_endpoint_residual_(1_minus_t)_velocity",
                "mse_random_interval_dt_velocity",
            ],
            "dt_min": DT_MIN,
            "dt_max": DT_MAX,
        },
    },
    checkpoint_path,
)

## Load Three Saved Models

Run this after initializing the three identical architectures. It restores all objective variants so the shared sampling and solvability comparison can run without retraining.


In [7]:
checkpoint_path = PROJECT_ROOT / "artifacts" / "maze_flow_matching_objectives.pt"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
for name, candidate in models.items():
    candidate.load_state_dict(checkpoint["models_state_dict"][name])
    candidate.eval()
binary_embeddings = checkpoint["binary_embeddings"].to(DEVICE)

## 6. ODE Sampling and Solvability Comparison

Sample each trained model from the same Gaussian starting states, integrate the velocity field with explicit Euler steps, and compare the fraction of generated mazes that are solvable. Intermediate trajectory images show the conventional velocity-MSE model.


In [12]:
N_METRIC_SAMPLES = 128
N_TO_SHOW = 4
ODE_STEPS = 100
PREVIEW_STEP_INDICES = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


@torch.no_grad()
def sample_flow(
    model, n_samples, steps=ODE_STEPS, capture_indices=(), initial_state=None
):
    model.eval()
    if initial_state is None:
        x = torch.randn(n_samples, EMBEDDING_DIM, 32, 32, device=DEVICE)
    else:
        if initial_state.shape != (n_samples, EMBEDDING_DIM, 32, 32):
            raise ValueError("initial_state has the wrong shape")
        x = initial_state.detach().clone()
    snapshots = {0: x.detach().clone()} if 0 in capture_indices else {}
    dt = 1.0 / steps
    for index in range(steps):
        t = index / steps
        times = torch.full((n_samples,), t, device=DEVICE)
        x = x + dt * model(x, times * TIME_EMBEDDING_SCALE)
        if index + 1 in capture_indices:
            snapshots[index + 1] = x.detach().clone()
    return x, snapshots


comparison_generator = torch.Generator(device=DEVICE).manual_seed(SEED + 501)
comparison_initial_noise = torch.randn(
    N_METRIC_SAMPLES,
    EMBEDDING_DIM,
    32,
    32,
    generator=comparison_generator,
    device=DEVICE,
    dtype=binary_embeddings.dtype,
)
comparison_samples = {}
for method_name, candidate in models.items():
    continuous, snapshots = sample_flow(
        candidate,
        N_METRIC_SAMPLES,
        capture_indices=PREVIEW_STEP_INDICES,
        initial_state=comparison_initial_noise,
    )
    mazes_binary = (
        torch.einsum("bdhw,kd->bkhw", continuous, binary_embeddings)
        .argmax(dim=1)
        .cpu()
        .numpy()
        .astype(np.uint8)
    )
    solvable = np.asarray([solvability_rate([maze]) == 1.0 for maze in mazes_binary])
    comparison_samples[method_name] = {
        "continuous": continuous,
        "snapshots": snapshots,
        "mazes": mazes_binary,
        "solvable": solvable,
    }

for method_name, candidate in models.items():
    snapshots = comparison_samples[method_name]["snapshots"]
    fig = make_subplots(
        rows=N_TO_SHOW,
        cols=len(PREVIEW_STEP_INDICES),
        subplot_titles=[
            f"t={index / ODE_STEPS:g}"
            for _ in range(N_TO_SHOW)
            for index in PREVIEW_STEP_INDICES
        ],
    )
    for row in range(N_TO_SHOW):
        for col, step_index in enumerate(PREVIEW_STEP_INDICES, start=1):
            state = snapshots[step_index][row : row + 1]
            probability_one = decode_probabilities(
                state, binary_embeddings, TEMPERATURE
            )[0, 1]
            fig.add_trace(
                go.Heatmap(
                    z=probability_one.cpu().numpy(),
                    zmin=0,
                    zmax=1,
                    colorscale=[[0, "black"], [1, "white"]],
                    showscale=False,
                ),
                row=row + 1,
                col=col,
            )
    fig.update_layout(
        title=f"{MODEL_LABELS[method_name]} - ODE trajectories",
        height=850,
        width=1900,
    )
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False, autorange="reversed")
    fig.show()

method_names = list(models)
solvability_rates = np.asarray(
    [comparison_samples[name]["solvable"].mean() for name in method_names]
)
solvability_intervals = []
for rate in solvability_rates:
    n = N_METRIC_SAMPLES
    z_score = 1.959963984540054
    denom = 1.0 + z_score**2 / n
    center = (rate + z_score**2 / (2 * n)) / denom
    radius = z_score * np.sqrt(rate * (1 - rate) / n + z_score**2 / (4 * n**2)) / denom
    solvability_intervals.append((max(0.0, center - radius), min(1.0, center + radius)))
solvability_intervals = np.asarray(solvability_intervals)
solvability_fig = go.Figure(
    go.Bar(
        x=[MODEL_LABELS[name] for name in method_names],
        y=solvability_rates,
        marker_color=[method_colors[name] for name in method_names],
        text=[f"{rate:.1%}" for rate in solvability_rates],
        textposition="outside",
        error_y=dict(
            type="data",
            symmetric=False,
            array=solvability_intervals[:, 1] - solvability_rates,
            arrayminus=solvability_rates - solvability_intervals[:, 0],
            color="#334155",
            thickness=1.5,
            width=5,
        ),
        customdata=np.column_stack(
            [solvability_intervals[:, 0], solvability_intervals[:, 1]]
        ),
        hovertemplate=(
            "%{x}<br>solvability=%{y:.1%}"
            "<br>95% Wilson interval=[%{customdata[0]:.1%}, %{customdata[1]:.1%}]<extra></extra>"
        ),
    )
)
solvability_fig.update_layout(
    title=f"Generated-maze solvability from {N_METRIC_SAMPLES} shared Gaussian starts",
    xaxis_title="Training objective",
    yaxis=dict(title="Solvability probability", range=[0, 1]),
    template="plotly_white",
    width=1000,
    height=560,
    showlegend=False,
)
solvability_fig.show()

for method_name in models:
    hard_samples = comparison_samples[method_name]["mazes"]
    solved = comparison_samples[method_name]["solvable"]
    hard_fig = make_subplots(
        rows=1,
        cols=N_TO_SHOW,
        subplot_titles=[
            f"Sample {i + 1}: {'solvable' if solved[i] else 'unsolvable'}"
            for i in range(N_TO_SHOW)
        ],
    )
    for i in range(N_TO_SHOW):
        hard_fig.add_trace(
            go.Heatmap(
                z=hard_samples[i],
                zmin=0,
                zmax=1,
                colorscale=[[0, "black"], [1, "white"]],
                showscale=False,
            ),
            row=1,
            col=i + 1,
        )
    hard_fig.update_layout(
        title=f"{MODEL_LABELS[method_name]} - final maze samples",
        height=390,
        width=900,
    )
    hard_fig.update_xaxes(showticklabels=False)
    hard_fig.update_yaxes(showticklabels=False, autorange="reversed")
    hard_fig.show()

## 7. Temperature calibration and adaptive integration

This section learns a temperature schedule for decoding continuous embeddings into pixel probabilities, then runs the reverse-entropy and adaptive-sampling diagnostics separately for each trained model using the shared learned temperature schedule.

1. **Calibrate the decoder.** Fit $T(t)$ so the mean forward-path pixel entropy follows the target curve.
2. **Check the schedule.** Compare the target against independent Monte Carlo estimates and forward/reverse maze trajectories.
3. **Adapt the sampler.** During reverse integration, evaluate candidate step sizes and accept a step when its predicted entropy reaches the local target.

Run sections 1 and 2 first. The temperature schedule can be fitted without the maze velocity model; load or train that model before checking reverse trajectories and running adaptive sampling.


### 7.1 Learn and validate the temperature schedule

#### Goal

At time $t$, the continuous state is decoded into probabilities for the two maze classes. The temperature controls how concentrated those probabilities are. Fit a schedule $T(t)$ so the **mean pixel entropy** of the forward process follows

$$
h_{\mathrm{target}}(t) = (1-t)\ln 2.
$$

This target starts at one bit of uncertainty per pixel at $t=0$ and reaches zero at $t=1$.

#### Parameterization and training

A small MLP takes $t$, $\log t$, and $\log(1-t)$ as input and predicts $\log T(t)$. The maze embeddings stay fixed: the network learns only the decoder temperature, not new embeddings or a different noise process.

For each training time, the expected entropy is estimated by Monte Carlo over Gaussian embedding noise. The loss matches entropy normalized by $\ln 2$ to the linear target $1-t$. Fresh antithetic noise draws are used during training, and the loss plot updates in place.

#### Validation and visual checks

After fitting, the schedule is checked using independent Gaussian draws, forward trajectories through 10 held-out mazes, and 10 reverse trajectories from Gaussian noise when the velocity model is available. The cell plots the learned temperature and entropy curves, then previews a forward trajectory with the same probability decoder. The endpoint limits are set explicitly: $T(0)=\infty$ and $T(1)=0$.


In [13]:
# Train a small temperature network against the actual forward entropy.

SCHEDULE_STEPS = 2000
SCHEDULE_LR = 3e-3
SCHEDULE_TIME_BATCH = 64
SCHEDULE_TRAIN_DRAWS = 2048  # Even: antithetic Gaussian pairs.
SCHEDULE_FINAL_DRAWS = 65536
SCHEDULE_PLOT_EVERY = 100
SCHEDULE_N_MAZES = 10
SCHEDULE_TIME_EPS = 1e-4
schedule_device = binary_embeddings.device
schedule_dtype = binary_embeddings.dtype
schedule_ln2 = float(np.log(2.0))

# Keep the maze model and its optimizer separate from the schedule network.
with torch.random.fork_rng(devices=[]):
    torch.random.default_generator.manual_seed(SEED + 702)
    temperature_net = torch.nn.Sequential(
        torch.nn.Linear(3, 32),
        torch.nn.SiLU(),
        torch.nn.Linear(32, 32),
        torch.nn.SiLU(),
        torch.nn.Linear(32, 1),
    ).to(device=schedule_device, dtype=schedule_dtype)
schedule_optimizer = torch.optim.Adam(temperature_net.parameters(), lr=SCHEDULE_LR)
schedule_train_rng = torch.Generator(device=schedule_device).manual_seed(SEED + 703)
schedule_final_rng = torch.Generator(device=schedule_device).manual_seed(SEED + 705)
schedule_maze_rng = torch.Generator(device=schedule_device).manual_seed(SEED + 706)

schedule_embeddings = binary_embeddings.detach()
schedule_cross = (schedule_embeddings[0] * schedule_embeddings[1]).sum()
schedule_clean_margins = schedule_embeddings.square().sum(dim=1) - schedule_cross
schedule_noise_std = (schedule_embeddings[1] - schedule_embeddings[0]).norm()
schedule_class_weights = torch.stack(
    [(val_mazes == pixel_class).float().mean() for pixel_class in (0, 1)]
).to(device=schedule_device, dtype=schedule_dtype)


def schedule_log_temperature(times):
    interior = times.clamp(SCHEDULE_TIME_EPS, 1.0 - SCHEDULE_TIME_EPS)
    features = torch.stack([interior, interior.log(), torch.log1p(-interior)], dim=-1)
    return temperature_net(features).squeeze(-1).clamp(-15.0, 15.0)


def schedule_binary_entropy(scaled_margin):
    # Stable at large logits: no log(0) or subtraction from sigmoid near 1.
    magnitude = scaled_margin.abs()
    return F.softplus(-magnitude) + magnitude * torch.sigmoid(-magnitude)


def schedule_mc_entropy(times, gaussian_draws):
    margins = (
        times[:, None, None] * schedule_clean_margins[None, :, None]
        + (1.0 - times[:, None, None])
        * schedule_noise_std
        * gaussian_draws[None, None, :]
    )
    temperature = schedule_log_temperature(times).exp()
    entropy_by_class = schedule_binary_entropy(
        margins / temperature[:, None, None]
    ).mean(dim=-1)
    mean_entropy = (entropy_by_class * schedule_class_weights[None, :]).sum(dim=-1)
    mean_entropy = torch.where(times <= 0, schedule_ln2, mean_entropy)
    return torch.where(times >= 1, 0.0, mean_entropy)


@torch.no_grad()
def learned_temperature(t):
    """Reusable scalar temperature schedule; endpoint limits are exact."""
    if t <= 0.0:
        return float("inf")
    if t >= 1.0:
        return 0.0
    time = torch.tensor([t], device=schedule_device, dtype=schedule_dtype)
    return schedule_log_temperature(time).exp().item()


@torch.no_grad()
def decode_with_learned_temperature(states, t):
    """One shared decoder for the entropy check and forward-path preview."""
    if t <= 0.0:
        return torch.full(
            (states.shape[0], 2, *states.shape[2:]),
            0.5,
            device=states.device,
            dtype=states.dtype,
        )
    if t >= 1.0:
        logits = torch.einsum("bdhw,kd->bkhw", states, binary_embeddings)
        maxima = (logits == logits.amax(dim=1, keepdim=True)).to(states.dtype)
        return maxima / maxima.sum(dim=1, keepdim=True)
    return decode_probabilities(states, binary_embeddings, learned_temperature(t))


schedule_check_times = torch.linspace(
    0, 1, 101, device=schedule_device, dtype=schedule_dtype
)
schedule_times_np = schedule_check_times.cpu().numpy()
schedule_target_np = (1.0 - schedule_times_np) * schedule_ln2
schedule_steps_history, schedule_loss_history = [], []

# Display one widget once; mutate its trace during training.
schedule_loss_fig = go.FigureWidget(
    go.Scatter(
        x=[],
        y=[],
        mode="lines",
        name="Training MSE (entropy / ln(2))",
    )
)
schedule_loss_fig.update_layout(
    title=f"Temperature MLP loss: step 0/{SCHEDULE_STEPS}",
    xaxis_title="Optimization step",
    yaxis_title="Normalized entropy MSE",
    yaxis_type="log",
    width=1000,
    height=420,
    showlegend=False,
)
display(schedule_loss_fig)
for step in range(1, SCHEDULE_STEPS + 1):
    temperature_net.train()
    # Cover both the interior and the neighborhoods of the endpoints.
    half_batch = SCHEDULE_TIME_BATCH // 2
    uniform_times = torch.rand(
        half_batch,
        generator=schedule_train_rng,
        device=schedule_device,
        dtype=schedule_dtype,
    )
    logit_extent = float(np.log((1 - SCHEDULE_TIME_EPS) / SCHEDULE_TIME_EPS))
    endpoint_times = torch.sigmoid(
        (
            2
            * torch.rand(
                SCHEDULE_TIME_BATCH - half_batch,
                generator=schedule_train_rng,
                device=schedule_device,
                dtype=schedule_dtype,
            )
            - 1
        )
        * logit_extent
    )
    train_times = torch.cat([uniform_times, endpoint_times]).clamp(
        SCHEDULE_TIME_EPS, 1.0 - SCHEDULE_TIME_EPS
    )
    half_noise = torch.randn(
        SCHEDULE_TRAIN_DRAWS // 2,
        generator=schedule_train_rng,
        device=schedule_device,
        dtype=schedule_dtype,
    )
    train_noise = torch.cat([half_noise, -half_noise])
    expected_entropy = schedule_mc_entropy(train_times, train_noise)
    normalized_error = expected_entropy / schedule_ln2 - (1.0 - train_times)
    schedule_loss = normalized_error.square().mean()
    schedule_optimizer.zero_grad(set_to_none=True)
    schedule_loss.backward()
    schedule_optimizer.step()
    schedule_steps_history.append(step)
    schedule_loss_history.append(schedule_loss.item())

    if step == 1 or step % SCHEDULE_PLOT_EVERY == 0 or step == SCHEDULE_STEPS:
        with schedule_loss_fig.batch_update():
            schedule_loss_fig.data[0].x = list(schedule_steps_history)
            schedule_loss_fig.data[0].y = np.maximum(schedule_loss_history, 1e-12)
            schedule_loss_fig.layout.title.text = (
                f"Temperature MLP loss: step {step}/{SCHEDULE_STEPS}"
            )


# After fitting: validate with fresh Gaussian draws and actual maze trajectories.
temperature_net.eval()
with torch.no_grad():
    final_noise = torch.randn(
        SCHEDULE_FINAL_DRAWS,
        generator=schedule_final_rng,
        device=schedule_device,
        dtype=schedule_dtype,
    )
    final_mc_entropy = (
        torch.cat(
            [
                schedule_mc_entropy(chunk, final_noise)
                for chunk in schedule_check_times.split(8)
            ]
        )
        .cpu()
        .numpy()
    )

    # Second check: actual 32x32 embedding trajectories, with fresh independent noise.
    schedule_clean = embed_binary(
        val_mazes[:SCHEDULE_N_MAZES].to(schedule_device), binary_embeddings
    )
    schedule_noise = torch.randn(
        schedule_clean.shape,
        generator=schedule_maze_rng,
        device=schedule_device,
        dtype=schedule_dtype,
    )
    schedule_maze_entropy = np.empty((schedule_clean.shape[0], len(schedule_times_np)))
    for time_index, t in enumerate(schedule_times_np):
        states = (1.0 - float(t)) * schedule_noise + float(t) * schedule_clean
        probabilities = decode_with_learned_temperature(states, float(t))
        pixel_entropy = -(
            probabilities
            * probabilities.clamp_min(torch.finfo(probabilities.dtype).tiny).log()
        ).sum(dim=1)
        schedule_maze_entropy[:, time_index] = pixel_entropy.mean((1, 2)).cpu().numpy()

# The forward path is shared; evaluate reverse trajectories for each model separately.
schedule_reverse_entropy_by_model = {}
for method_name, candidate in models.items():
    reverse_rng = torch.Generator(device=schedule_device).manual_seed(SEED + 707)
    candidate.eval()
    with torch.no_grad():
        reverse_state = torch.randn(
            schedule_clean.shape,
            generator=reverse_rng,
            device=schedule_device,
            dtype=schedule_dtype,
        )
        reverse_entropy = np.empty_like(schedule_maze_entropy)
        for time_index, t in enumerate(schedule_times_np):
            if time_index > 0:
                previous_t = float(schedule_times_np[time_index - 1])
                dt = float(t) - previous_t
                model_times = torch.full(
                    (reverse_state.shape[0],),
                    previous_t * TIME_EMBEDDING_SCALE,
                    device=schedule_device,
                    dtype=schedule_dtype,
                )
                velocity = candidate(reverse_state, model_times)
                reverse_state = reverse_state + dt * velocity
            probabilities = decode_with_learned_temperature(reverse_state, float(t))
            pixel_entropy = -(
                probabilities
                * probabilities.clamp_min(torch.finfo(probabilities.dtype).tiny).log()
            ).sum(dim=1)
            reverse_entropy[:, time_index] = pixel_entropy.mean((1, 2)).cpu().numpy()
    schedule_reverse_entropy_by_model[method_name] = reverse_entropy

    schedule_validation_fig = go.Figure()
    for sample_index, values in enumerate(schedule_maze_entropy):
        schedule_validation_fig.add_scatter(
            x=schedule_times_np,
            y=values,
            mode="lines",
            line=dict(color="rgba(45,120,210,0.25)", width=1),
            name="Forward maze trajectories",
            showlegend=sample_index == 0,
        )
    schedule_validation_fig.add_scatter(
        x=schedule_times_np,
        y=schedule_maze_entropy.mean(axis=0),
        name=f"Forward mean ({schedule_clean.shape[0]} trajectories)",
        line=dict(color="royalblue", width=3),
    )
    for sample_index, values in enumerate(reverse_entropy):
        schedule_validation_fig.add_scatter(
            x=schedule_times_np,
            y=values,
            mode="lines",
            line=dict(color="rgba(35,160,95,0.22)", width=1),
            name="Reverse trajectories",
            showlegend=sample_index == 0,
        )
    schedule_validation_fig.add_scatter(
        x=schedule_times_np,
        y=reverse_entropy.mean(axis=0),
        name="Reverse mean",
        line=dict(color="seagreen", width=3),
    )
    schedule_validation_fig.add_scatter(
        x=schedule_times_np,
        y=final_mc_entropy,
        name=f"Independent Gaussian MC ({SCHEDULE_FINAL_DRAWS:,} draws)",
        line=dict(color="orangered", width=2),
    )
    schedule_validation_fig.add_scatter(
        x=schedule_times_np,
        y=schedule_target_np,
        name="Linear target",
        line=dict(color="black", dash="dash", width=2),
    )
    schedule_validation_fig.update_layout(
        title=f"{MODEL_LABELS[method_name]} - forward and reverse entropy",
        xaxis_title="t",
        yaxis_title="Mean pixel entropy (nats)",
        width=1150,
        height=520,
    )
    schedule_validation_fig.update_xaxes(range=[0, 1])
    schedule_validation_fig.update_yaxes(range=[0, schedule_ln2 * 1.05])
    schedule_validation_fig.show()

schedule_temperature_fig = go.Figure(
    go.Scatter(
        x=schedule_times_np[1:-1],
        y=[learned_temperature(float(t)) for t in schedule_times_np[1:-1]],
        name="Learned T(t)",
        mode="lines",
    )
)
schedule_temperature_fig.update_layout(
    title="Temperature learned by the MLP (endpoints excluded)",
    xaxis_title="t",
    yaxis_title="T(t)",
    yaxis_type="log",
    width=1000,
    height=400,
)
schedule_temperature_fig.show()
# Finally, show one of the same independent forward trajectories used above.
SCHEDULE_PREVIEW_TIMES = [0.0, 0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0]
schedule_forward_fig = make_subplots(
    rows=1,
    cols=len(SCHEDULE_PREVIEW_TIMES),
    subplot_titles=[f"t={t:g}" for t in SCHEDULE_PREVIEW_TIMES],
    horizontal_spacing=0.008,
)
with torch.no_grad():
    for col, t in enumerate(SCHEDULE_PREVIEW_TIMES, start=1):
        state = (1.0 - t) * schedule_noise[:1] + t * schedule_clean[:1]
        probability_one = decode_with_learned_temperature(state, t)[0, 1]
        schedule_forward_fig.add_trace(
            go.Heatmap(
                z=probability_one.cpu().numpy(),
                zmin=0,
                zmax=1,
                colorscale=[[0, "black"], [1, "white"]],
                showscale=False,
                zsmooth=False,
            ),
            row=1,
            col=col,
        )
        schedule_forward_fig.update_yaxes(
            scaleanchor="x" if col == 1 else f"x{col}",
            scaleratio=1,
            row=1,
            col=col,
        )
schedule_forward_fig.update_layout(
    title="Forward process with learned T(t): black = p(0), white = p(1)",
    width=1700,
    height=320,
)
schedule_forward_fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
schedule_forward_fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    zeroline=False,
    autorange="reversed",
)
schedule_forward_fig.show()

FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'Training MSE (entropy / ln(2))',
              'type': 'scatter',
              'uid': 'fa79d459-3ec1-41f8-8a74-fc9402ead4be',
              'x': [],
              'y': []}],
    'layout': {'height': 420,
               'showlegend': False,
               'template': '...',
               'title': {'text': 'Temperature MLP loss: step 0/2000'},
               'width': 1000,
               'xaxis': {'title': {'text': 'Optimization step'}},
               'yaxis': {'title': {'text': 'Normalized entropy MSE'}, 'type': 'log'}}
})

### 7.2 Reverse sampling with adaptive time steps

The next cell runs the same adaptive sampler separately for each trained velocity model. Each model receives the same Gaussian starting state and gets its own labeled video, using the shared learned temperature schedule and the configured entropy threshold.


In [15]:
# Adaptive integration and a prerecorded video of its variable-length steps.
ADAPTIVE_DT = 0.01
ADAPTIVE_ENTROPY_THRESHOLD = 0.002  # Signed residual, in nats; not absolute error.
ADAPTIVE_NEIGHBOURHOOD_THRESHOLD = 0.02
ADAPTIVE_VIDEO_FPS = 24
ADAPTIVE_VIDEO_WIDTH = 1920
ADAPTIVE_VIDEO_HEIGHT = 1080
ADAPTIVE_TRACE_SPEED_PIXELS_PER_SECOND = 520
ADAPTIVE_HOLD_SECONDS = 1.0


@torch.no_grad()
def adaptive_mean_entropy(state, t):
    probabilities = decode_with_learned_temperature(state, t)
    log_p = probabilities.clamp_min(torch.finfo(probabilities.dtype).tiny).log()
    return -(probabilities * log_p).sum(dim=1).mean().item()


@torch.no_grad()
def adaptive_entropy_sample(
    candidate_model,
    initial_state,
    dt=ADAPTIVE_DT,
    entropy_threshold=ADAPTIVE_ENTROPY_THRESHOLD,
    neighbourhood_threshold=ADAPTIVE_NEIGHBOURHOOD_THRESHOLD,
    visualize=False,
    return_iterations=False,
):
    """Return a sample and optionally its iteration count or visualization history."""
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("dt must be finite and positive.")
    if not np.isfinite(entropy_threshold):
        raise ValueError("entropy_threshold must be finite.")
    if not 0 < neighbourhood_threshold < 1:
        raise ValueError("neighbourhood_threshold must lie between 0 and 1.")
    if initial_state.ndim != 4 or initial_state.shape[0] != 1:
        raise ValueError("Use one initial state with shape (1, D, H, W).")

    prev_pred = initial_state.detach().clone()
    t0 = 0.0
    records = []
    prior_stops = []
    if visualize:
        initial_image = (
            decode_with_learned_temperature(prev_pred, 0.0)[0, 1].cpu().numpy()
        )
        records.append(
            dict(
                iteration=0,
                t0=0.0,
                h0=float(np.log(2.0)),
                t_next=0.0,
                terminal=False,
                scan_t=np.array([0.0]),
                scan_h=np.array([np.log(2.0)]),
                stop_index=0,
                stop_h=float(np.log(2.0)),
                prior_stops=[],
                before_image=initial_image,
                after_image=initial_image,
            )
        )

    was_training = candidate_model.training
    candidate_model.eval()
    try:
        # Every ordinary update advances by dt or ends at t=1.
        max_updates = int(np.ceil(1.0 / dt)) + 2
        for iteration in range(1, max_updates + 1):
            # Anchor this iteration's target to the entropy of its current state.
            h0 = (
                float(np.log(2.0))
                if t0 == 0.0
                else adaptive_mean_entropy(prev_pred, t0)
            )
            target_slope = h0 / (1.0 - t0)
            model_time = torch.full(
                (1,),
                t0 * TIME_EMBEDDING_SCALE,
                device=prev_pred.device,
                dtype=prev_pred.dtype,
            )
            predicted_velocity = candidate_model(prev_pred, model_time)
            terminal = t0 >= 1.0 - neighbourhood_threshold
            before_image = (
                decode_with_learned_temperature(prev_pred, t0)[0, 1].cpu().numpy()
                if visualize
                else None
            )
            scan_t = [t0] if visualize else []
            scan_h = [h0] if visualize else []
            stop_index = None
            t_next = None
            n_candidates = max(1, int(np.ceil((1.0 - t0) / dt)))

            if terminal and not visualize:
                # The terminal rule requires no candidate scan.
                next_pred = prev_pred + (1.0 - t0) * predicted_velocity
                return (next_pred, iteration) if return_iterations else next_pred

            for n in range(1, n_candidates + 1):
                t1 = 1.0 if n == n_candidates else min(t0 + n * dt, 1.0)
                curr_pred = prev_pred + (t1 - t0) * predicted_velocity
                entropy = adaptive_mean_entropy(curr_pred, t1)
                if visualize:
                    scan_t.append(t1)
                    scan_h.append(entropy)
                target_entropy = (1.0 - t1) * target_slope
                residual = entropy - target_entropy
                if t_next is None and (
                    (not terminal and residual >= entropy_threshold) or t1 >= 1.0
                ):
                    t_next = t1
                    stop_index = n
                    if not visualize:
                        break
                # With visualization enabled, finish the curve with this same velocity.

            next_pred = (
                prev_pred + (1.0 - t0) * predicted_velocity
                if terminal
                else prev_pred + (t_next - t0) * predicted_velocity
            )
            if visualize:
                stop_h = adaptive_mean_entropy(next_pred, t_next)
                records.append(
                    dict(
                        iteration=iteration,
                        t0=t0,
                        h0=h0,
                        t_next=t_next,
                        terminal=terminal,
                        scan_t=np.asarray(scan_t),
                        scan_h=np.asarray(scan_h),
                        stop_index=stop_index,
                        stop_h=stop_h,
                        prior_stops=list(prior_stops),
                        before_image=before_image,
                        after_image=decode_with_learned_temperature(next_pred, t_next)[
                            0, 1
                        ]
                        .cpu()
                        .numpy(),
                    )
                )
                prior_stops.append((t_next, stop_h))
            prev_pred = next_pred
            t0 = t_next
            if t0 >= 1.0:
                if visualize:
                    return prev_pred, records
                return (prev_pred, iteration) if return_iterations else prev_pred
        raise RuntimeError("Adaptive integration exceeded its progress bound.")
    finally:
        candidate_model.train(was_training)


from IPython.display import Video, Markdown
import _shared.adaptive_video as adaptive_video_module

importlib.reload(adaptive_video_module)
from _shared.adaptive_video import render_adaptive_video

adaptive_initial_state = torch.randn(
    1,
    binary_embeddings.shape[1],
    32,
    32,
    device=binary_embeddings.device,
    dtype=binary_embeddings.dtype,
    generator=torch.Generator(device=DEVICE).manual_seed(SEED + 601),
)
adaptive_video_paths = {}
adaptive_histories = {}
for method_name, candidate in models.items():
    display(Markdown(f"### {MODEL_LABELS[method_name]}"))
    adaptive_sample, adaptive_history = adaptive_entropy_sample(
        candidate,
        adaptive_initial_state.clone(),
        visualize=True,
    )
    adaptive_histories[method_name] = adaptive_history
    video_path = render_adaptive_video(
        adaptive_history,
        PROJECT_ROOT / "artifacts" / f"adaptive_sampling_{method_name}.mp4",
        fps=ADAPTIVE_VIDEO_FPS,
        width=ADAPTIVE_VIDEO_WIDTH,
        height=ADAPTIVE_VIDEO_HEIGHT,
        trace_speed_pixels_per_second=ADAPTIVE_TRACE_SPEED_PIXELS_PER_SECOND,
        hold_seconds=ADAPTIVE_HOLD_SECONDS,
    )
    adaptive_video_paths[method_name] = video_path
    display(
        Video(
            str(video_path),
            embed=True,
            html_attributes=(
                'controls autoplay muted preload="auto" playsinline '
                'style="display:block;width:100%;max-width:100%;height:auto;border-radius:12px"'
            ),
        )
    )

### 0: Velocity MSE

### 1: Endpoint-residual MSE

### 2: Random-interval MSE

## 8. Adaptive steps versus fixed steps

For each trained model, the first Monte Carlo sweep estimates iteration-count and threshold distributions. The next cell fits that model?s inverse calibration and compares its adaptive and fixed-step maze solvability at matched iteration budgets.

Each paired trial uses the same Gaussian starting state for both solvers. The adaptive solver receives the threshold predicted for the target iteration count. The fixed-step solver uses a constant Euler step within each sample; stochastic rounding between neighboring integer step counts matches a fractional target budget. The solvability plot shows Monte Carlo means with asymmetric 95% Wilson intervals.


In [16]:
# Monte Carlo adaptive-step distributions, repeated for every trained model.
ADAPTIVE_ITERATION_MC_THRESHOLDS = np.geomspace(0.001, 0.1, 10)
ADAPTIVE_ITERATION_MC_REPEATS = 100

import ipywidgets as widgets

iteration_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=len(models)
    * len(ADAPTIVE_ITERATION_MC_THRESHOLDS)
    * ADAPTIVE_ITERATION_MC_REPEATS,
    description="Adaptive MC",
    bar_style="info",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
display(iteration_progress)
iteration_progress_interval = max(1, iteration_progress.max // 100)
iteration_completed = 0
iteration_rng = torch.Generator(device=DEVICE).manual_seed(SEED + 801)
iteration_initial_states = torch.randn(
    ADAPTIVE_ITERATION_MC_REPEATS,
    EMBEDDING_DIM,
    32,
    32,
    generator=iteration_rng,
    device=DEVICE,
    dtype=binary_embeddings.dtype,
)
adaptive_iteration_results = {}

for method_index, (method_name, candidate) in enumerate(models.items(), start=1):
    counts = np.empty(
        (len(ADAPTIVE_ITERATION_MC_THRESHOLDS), ADAPTIVE_ITERATION_MC_REPEATS),
        dtype=np.int32,
    )
    for threshold_index, threshold in enumerate(ADAPTIVE_ITERATION_MC_THRESHOLDS):
        iteration_progress.description = (
            f"{MODEL_LABELS[method_name]} - threshold "
            f"{threshold_index + 1}/{len(ADAPTIVE_ITERATION_MC_THRESHOLDS)}"
        )
        for sample_index in range(ADAPTIVE_ITERATION_MC_REPEATS):
            _, n_iterations = adaptive_entropy_sample(
                candidate,
                iteration_initial_states[sample_index : sample_index + 1],
                entropy_threshold=float(threshold),
                return_iterations=True,
            )
            counts[threshold_index, sample_index] = n_iterations
            iteration_completed += 1
            if (
                iteration_completed % iteration_progress_interval == 0
                or iteration_completed == iteration_progress.max
            ):
                iteration_progress.value = iteration_completed

    iteration_min = int(counts.min())
    iteration_max = int(counts.max())
    iteration_axis = np.arange(iteration_min, iteration_max + 1)
    iteration_pmf = np.zeros(
        (len(iteration_axis), len(ADAPTIVE_ITERATION_MC_THRESHOLDS)), dtype=float
    )
    for threshold_index, row in enumerate(counts):
        iteration_pmf[:, threshold_index] = np.bincount(
            row - iteration_min, minlength=len(iteration_axis)
        ) / len(row)
    iteration_mean = counts.mean(axis=1)
    iteration_q05, iteration_q95 = np.quantile(counts, [0.05, 0.95], axis=1)
    iteration_evidence = iteration_pmf.sum(axis=1)
    threshold_given_iterations = np.divide(
        iteration_pmf,
        iteration_evidence[:, None],
        out=np.zeros_like(iteration_pmf),
        where=iteration_evidence[:, None] > 0,
    )
    adaptive_iteration_results[method_name] = {
        "counts": counts,
        "iteration_axis": iteration_axis,
        "iteration_pmf": iteration_pmf,
        "iteration_mean": iteration_mean,
        "iteration_q05": iteration_q05,
        "iteration_q95": iteration_q95,
        "iteration_evidence": iteration_evidence,
        "threshold_given_iterations": threshold_given_iterations,
    }

    iteration_pdf_fig = go.Figure()
    iteration_pdf_fig.add_trace(
        go.Heatmap(
            x=ADAPTIVE_ITERATION_MC_THRESHOLDS,
            y=iteration_axis,
            z=iteration_pmf,
            colorscale="Blues",
            colorbar=dict(title="P(N)"),
            name="Iteration-count PMF",
            hovertemplate="threshold=%{x:.3g}<br>iterations=%{y}<br>P(N)=%{z:.3f}<extra></extra>",
        )
    )
    iteration_pdf_fig.add_trace(
        go.Scatter(
            x=ADAPTIVE_ITERATION_MC_THRESHOLDS,
            y=iteration_q05,
            yaxis="y2",
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        )
    )
    iteration_pdf_fig.add_trace(
        go.Scatter(
            x=ADAPTIVE_ITERATION_MC_THRESHOLDS,
            y=iteration_q95,
            yaxis="y2",
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(239, 125, 35, 0.18)",
            name="5th-95th percentile",
            hoverinfo="skip",
        )
    )
    iteration_pdf_fig.add_trace(
        go.Scatter(
            x=ADAPTIVE_ITERATION_MC_THRESHOLDS,
            y=iteration_mean,
            yaxis="y2",
            mode="lines+markers",
            line=dict(color="#c45b12", width=3),
            marker=dict(size=7),
            name="E[iterations]",
            hovertemplate="threshold=%{x:.3g}<br>E[N]=%{y:.2f}<extra></extra>",
        )
    )
    iteration_pdf_fig.update_layout(
        title=f"{MODEL_LABELS[method_name]} - adaptive iteration distribution",
        xaxis=dict(title="ADAPTIVE_ENTROPY_THRESHOLD", type="log"),
        yaxis=dict(
            title="Probability mass P(N | threshold)",
            range=[iteration_min - 0.5, iteration_max + 0.5],
        ),
        yaxis2=dict(
            title="Iteration count",
            overlaying="y",
            side="right",
            range=[iteration_min - 0.5, iteration_max + 0.5],
        ),
        template="plotly_white",
        width=1100,
        height=620,
        legend=dict(orientation="h", y=1.08),
    )
    iteration_pdf_fig.show()

    threshold_pdf_fig = go.Figure(
        go.Heatmap(
            x=iteration_axis,
            y=ADAPTIVE_ITERATION_MC_THRESHOLDS,
            z=threshold_given_iterations.T,
            colorscale="Viridis",
            colorbar=dict(title="P(threshold | N)"),
            hovertemplate=(
                "iterations=%{x}<br>threshold=%{y:.4g}"
                "<br>probability mass=%{z:.3f}<extra></extra>"
            ),
        )
    )
    threshold_pdf_fig.update_layout(
        title=f"{MODEL_LABELS[method_name]} - threshold distribution by iterations",
        xaxis_title="Adaptive iteration count N",
        yaxis=dict(title="ADAPTIVE_ENTROPY_THRESHOLD", type="log"),
        template="plotly_white",
        width=1100,
        height=620,
    )
    threshold_pdf_fig.show()

iteration_progress.bar_style = "success"
iteration_progress.description = "Adaptive MC complete"

IntProgress(value=0, bar_style='info', description='Adaptive MC', layout=Layout(width='100%'), max=3000, style…

In [17]:
# Invert each model's threshold/iteration calibration and compare its two solvers.
def posterior_threshold_quantile(probabilities, quantile):
    if probabilities.sum() <= 0:
        return np.nan
    index = int(np.searchsorted(np.cumsum(probabilities), quantile, side="left"))
    index = min(index, len(ADAPTIVE_ITERATION_MC_THRESHOLDS) - 1)
    return float(ADAPTIVE_ITERATION_MC_THRESHOLDS[index])


def isotonic_nonincreasing(values, weights):
    """Weighted pool-adjacent-violators fit constrained to be non-increasing."""
    blocks = []
    for index, (value, weight) in enumerate(zip(values, weights)):
        blocks.append([index, index + 1, float(weight), float(weight * value)])
        while len(blocks) >= 2:
            previous_mean = blocks[-2][3] / blocks[-2][2]
            current_mean = blocks[-1][3] / blocks[-1][2]
            if previous_mean >= current_mean:
                break
            right = blocks.pop()
            left = blocks.pop()
            blocks.append([left[0], right[1], left[2] + right[2], left[3] + right[3]])
    fitted = np.empty(len(values), dtype=float)
    for start, end, weight, weighted_sum in blocks:
        fitted[start:end] = weighted_sum / weight
    return fitted


def wilson_interval(successes, trials, z=1.959963984540054):
    p = successes / trials
    denominator = 1.0 + z**2 / trials
    center = (p + z**2 / (2.0 * trials)) / denominator
    radius = (
        z * np.sqrt(p * (1.0 - p) / trials + z**2 / (4.0 * trials**2)) / denominator
    )
    return max(0.0, center - radius), min(1.0, center + radius)


@torch.no_grad()
def fixed_step_sample(initial_state, n_steps, candidate):
    n_steps = max(1, int(n_steps))
    state = initial_state.detach().clone()
    step_size = 1.0 / n_steps
    for step_index in range(n_steps):
        t = step_index / n_steps
        time = torch.full(
            (state.shape[0],),
            t * TIME_EMBEDDING_SCALE,
            device=state.device,
            dtype=state.dtype,
        )
        state = state + step_size * candidate(state, time)
    return state


@torch.no_grad()
def hard_decode_maze(state):
    logits = torch.einsum("bdhw,kd->bkhw", state, binary_embeddings)
    return logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)


ADAPTIVE_BENCHMARK_BINS = 10
ADAPTIVE_BENCHMARK_REPEATS_PER_BIN = 100
benchmark_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=len(models) * ADAPTIVE_BENCHMARK_BINS * ADAPTIVE_BENCHMARK_REPEATS_PER_BIN,
    description="Solver comparison",
    bar_style="info",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
display(benchmark_progress)
benchmark_progress_interval = max(1, benchmark_progress.max // 100)
benchmark_completed = 0
adaptive_benchmark_results = {}

for model_index, (method_name, candidate) in enumerate(models.items()):
    results = adaptive_iteration_results[method_name]
    iteration_axis = results["iteration_axis"]
    iteration_pmf = results["iteration_pmf"]
    iteration_evidence = results["iteration_evidence"]
    threshold_given_iterations = results["threshold_given_iterations"]

    quantiles = np.full((len(iteration_axis), 3), np.nan)
    for iteration_index, probabilities in enumerate(threshold_given_iterations):
        quantiles[iteration_index] = [
            posterior_threshold_quantile(probabilities, q) for q in (0.10, 0.50, 0.90)
        ]
    valid_rows = np.isfinite(quantiles[:, 1])
    fit_iteration_values = iteration_axis[valid_rows].astype(float)
    fit_threshold_q10, fit_threshold_median, fit_threshold_q90 = quantiles[valid_rows].T
    fit_weights = iteration_evidence[valid_rows]
    fit_log_threshold = isotonic_nonincreasing(
        np.log(fit_threshold_median), fit_weights
    )
    fit_iteration_grid = np.linspace(
        float(fit_iteration_values.min()), float(fit_iteration_values.max()), 300
    )
    fit_threshold_grid = np.exp(
        np.interp(fit_iteration_grid, fit_iteration_values, fit_log_threshold)
    )

    def threshold_for_iterations(iterations):
        values = np.asarray(iterations, dtype=float)
        log_values = np.interp(values, fit_iteration_values, fit_log_threshold)
        return np.exp(log_values)

    inverse_fig = go.Figure()
    inverse_fig.add_trace(
        go.Scatter(
            x=fit_iteration_values,
            y=fit_threshold_q10,
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        )
    )
    inverse_fig.add_trace(
        go.Scatter(
            x=fit_iteration_values,
            y=fit_threshold_q90,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(66, 99, 235, 0.16)",
            name="Posterior 10th-90th percentile",
            hoverinfo="skip",
        )
    )
    inverse_fig.add_trace(
        go.Scatter(
            x=fit_iteration_values,
            y=fit_threshold_median,
            mode="markers",
            marker=dict(color="#4263eb", size=7),
            name="Posterior median threshold",
        )
    )
    inverse_fig.add_trace(
        go.Scatter(
            x=fit_iteration_grid,
            y=fit_threshold_grid,
            mode="lines",
            line=dict(color="#e76f00", width=3),
            name="Monotone isotonic fit",
        )
    )
    inverse_fig.update_layout(
        title=f"{MODEL_LABELS[method_name]} - threshold calibrated from iteration count",
        xaxis_title="Adaptive iteration count N",
        yaxis=dict(title="ADAPTIVE_ENTROPY_THRESHOLD", type="log"),
        template="plotly_white",
        width=1050,
        height=520,
        legend=dict(orientation="h", y=1.08),
    )
    inverse_fig.show()

    benchmark_iteration_edges = np.linspace(
        float(fit_iteration_values.min()),
        float(fit_iteration_values.max()),
        ADAPTIVE_BENCHMARK_BINS + 1,
    )
    benchmark_iteration_centers = 0.5 * (
        benchmark_iteration_edges[:-1] + benchmark_iteration_edges[1:]
    )
    benchmark_rng = np.random.default_rng(SEED + 802)
    benchmark_torch_rng = torch.Generator(device=DEVICE).manual_seed(SEED + 803)
    benchmark_solved = np.zeros(
        (2, ADAPTIVE_BENCHMARK_BINS, ADAPTIVE_BENCHMARK_REPEATS_PER_BIN), dtype=bool
    )
    benchmark_adaptive_iterations = np.zeros(
        (ADAPTIVE_BENCHMARK_BINS, ADAPTIVE_BENCHMARK_REPEATS_PER_BIN), dtype=np.int32
    )
    benchmark_fixed_iterations = np.zeros_like(benchmark_adaptive_iterations)

    for bin_index, (low, high) in enumerate(
        zip(benchmark_iteration_edges[:-1], benchmark_iteration_edges[1:])
    ):
        benchmark_progress.description = f"{MODEL_LABELS[method_name]} - budget {bin_index + 1}/{ADAPTIVE_BENCHMARK_BINS}"
        for repeat_index in range(ADAPTIVE_BENCHMARK_REPEATS_PER_BIN):
            target_iterations = float(benchmark_rng.uniform(low, high))
            threshold = float(threshold_for_iterations(target_iterations))
            initial_state = torch.randn(
                1,
                EMBEDDING_DIM,
                32,
                32,
                generator=benchmark_torch_rng,
                device=DEVICE,
                dtype=binary_embeddings.dtype,
            )
            adaptive_state, adaptive_n = adaptive_entropy_sample(
                candidate,
                initial_state,
                entropy_threshold=threshold,
                return_iterations=True,
            )
            lower_n = max(1, int(np.floor(target_iterations)))
            fixed_n = lower_n + int(
                benchmark_rng.random() < target_iterations - lower_n
            )
            fixed_state = fixed_step_sample(initial_state, fixed_n, candidate)
            benchmark_solved[0, bin_index, repeat_index] = (
                solvability_rate([hard_decode_maze(adaptive_state)]) == 1.0
            )
            benchmark_solved[1, bin_index, repeat_index] = (
                solvability_rate([hard_decode_maze(fixed_state)]) == 1.0
            )
            benchmark_adaptive_iterations[bin_index, repeat_index] = adaptive_n
            benchmark_fixed_iterations[bin_index, repeat_index] = fixed_n
            benchmark_completed += 1
            if (
                benchmark_completed % benchmark_progress_interval == 0
                or benchmark_completed == benchmark_progress.max
            ):
                benchmark_progress.value = benchmark_completed

    benchmark_method_names = ["Adaptive step", "Fixed step (matched E[N])"]
    benchmark_method_colors = ["#4263eb", "#e76f00"]
    benchmark_rates = benchmark_solved.mean(axis=2)
    benchmark_intervals = np.empty((2, ADAPTIVE_BENCHMARK_BINS, 2), dtype=float)
    for solver_index in range(2):
        for bin_index in range(ADAPTIVE_BENCHMARK_BINS):
            successes = int(benchmark_solved[solver_index, bin_index].sum())
            benchmark_intervals[solver_index, bin_index] = wilson_interval(
                successes, ADAPTIVE_BENCHMARK_REPEATS_PER_BIN
            )

    comparison_fig = go.Figure()
    for solver_index, solver_name in enumerate(benchmark_method_names):
        lower = benchmark_intervals[solver_index, :, 0]
        upper = benchmark_intervals[solver_index, :, 1]
        comparison_fig.add_trace(
            go.Scatter(
                x=benchmark_iteration_centers,
                y=lower,
                mode="lines",
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
            )
        )
        comparison_fig.add_trace(
            go.Scatter(
                x=benchmark_iteration_centers,
                y=upper,
                mode="lines",
                line=dict(width=0),
                fill="tonexty",
                fillcolor=(
                    "rgba(66, 99, 235, 0.15)"
                    if solver_index == 0
                    else "rgba(231, 111, 0, 0.15)"
                ),
                name=f"{solver_name}: 95% Wilson interval",
                hoverinfo="skip",
            )
        )
        comparison_fig.add_trace(
            go.Scatter(
                x=benchmark_iteration_centers,
                y=benchmark_rates[solver_index],
                mode="lines+markers",
                line=dict(color=benchmark_method_colors[solver_index], width=3),
                marker=dict(size=8),
                name=solver_name,
                customdata=np.column_stack([lower, upper]),
                hovertemplate=(
                    "E[N]=%{x:.2f}<br>solvability=%{y:.1%}"
                    "<br>95% interval=[%{customdata[0]:.1%}, %{customdata[1]:.1%}]<extra></extra>"
                ),
            )
        )
    comparison_fig.update_layout(
        title=f"{MODEL_LABELS[method_name]} - solvability vs iteration budget",
        xaxis_title="Target adaptive iterations N",
        yaxis=dict(title="Solvability probability", range=[0, 1]),
        template="plotly_white",
        width=1100,
        height=600,
        legend=dict(orientation="h", y=1.08),
    )
    comparison_fig.show()
    adaptive_benchmark_results[method_name] = {
        "solved": benchmark_solved,
        "adaptive_iterations": benchmark_adaptive_iterations,
        "fixed_iterations": benchmark_fixed_iterations,
        "iteration_centers": benchmark_iteration_centers,
        "rates": benchmark_rates,
        "intervals": benchmark_intervals,
    }

benchmark_progress.bar_style = "success"
benchmark_progress.description = "Solver comparison complete"

IntProgress(value=0, bar_style='info', description='Solver comparison', layout=Layout(width='100%'), max=3000,…